In [1]:
#!pip install sentence-transformers

What subset of transformers models would be best to try?

How about a set of experiments with different number length too. So dataset1 includes numbers 0-10, dataset2 0-100, and so on.
Then, let's compare performance between number vs spell-out.
- I think it also had trouble with floats?


I'm thinking... in the pipeline of experiments... apart from different transformers, maybe distances to the embeddings can be tested? Like the method for each, like L2 and other distances ETC.

BERT-base-uncased collapses to 0 for everything except 30, which it guesses correctly.
BERT-base-cased collapses to 52 for everything.

# Cleaned notebook

In [2]:
import pathlib
from dataset_writing import *
from experiment import run_experiment
from plots import *
from models import SentenceEncoder
import pandas as pd

Wrote cleaned dataset to ..\data\cleaned_data.csv


In [6]:
MODELS = {
    "MiniLM-L6": "all-MiniLM-L6-v2",
    "mpnet": "all-mpnet-base-v2",
    "BERT-nli-mean-tokens": "bert-base-nli-mean-tokens",
    "BERT-base-uncased": "bert-base-uncased",
    "BERT-base-cased": "bert-base-cased",
    "RoBERTa": "all-roberta-large-v1",
    "MiniLM-L12": "all-MiniLM-12-v2",
    "e5-small": "intfloat/e5-small",
    "unsup-simcse-bert": "princeton-nlp/unsup-simcse-bert-base-uncased",
    "MathBERT": "mathbert-base-uncased"
}

In [7]:
model_name = MODELS["MathBERT"]
#DATASET_FILE = pathlib.Path("../data/ages.txt")
AGE_RANGE = range(0, 100)
PCA_DIMS = 30
TSNE_PERPLEXITY = 30 # check the method to define the best perplexity.

In [8]:
encoder = SentenceEncoder(model_name)

No sentence-transformers model found with name sentence-transformers/mathbert-base-uncased. Creating a new one with mean pooling.


OSError: sentence-transformers/mathbert-base-uncased is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
age_format = "digits"
DATASET_FILE = pathlib.Path(f"../data/ages_{age_format}.txt")
#DATASET_FILE = pathlib.Path(f"../data/cleaned_data.csv")
out_dir = pathlib.Path(f"../results/{age_format}/{model_name}")
#out_dir = pathlib.Path(f"../results/bigger_dataset/{model_name}")
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
#generate_sentences(DATASET_FILE, style=age_format) # already generated.

In [ ]:
#sentences = load_sentences(path=DATASET_FILE)
sentences = load_bigger_dataset(path=DATASET_FILE)

In [ ]:
results = run_experiment(encoder, sentences, ages = AGE_RANGE)

In [ ]:
df_points = pd.DataFrame({
    "age": list(AGE_RANGE),
    "true": results["true"],
    "pred": results["pred"],
    "error": results["errors"],
    "within_+-2": results["correct"],
    "full_error": results["full_error"],
})

safe_name = model_name.replace("/", "-")
csv_path = out_dir / f"{safe_name}_results.csv"
df_points.to_csv(csv_path, index=False)
print("Saved results to {}".format(csv_path))

In [ ]:
summary = pd.DataFrame([{
    "model": model_name,
    "accuracy": results["accuracy"],
    "exact_match_acc": results["exact_match_acc"]
}])

summary_path = out_dir / f"{safe_name}_summary.csv"
summary.to_csv(summary_path, index=False)

print("Summary saved to {}".format(summary_path))

In [ ]:
print(results["exact_match_acc"])

In [ ]:
print(results["accuracy"])

In [ ]:
# If you want to re-do the graphs without having to re-run the experiments.

"""df_points  = pd.read_csv(csv_path)

results = {
    "true":       df_points["true"].tolist(),
    "pred":       df_points["pred"].tolist(),
    "errors":     df_points["error"].tolist(),
    "correct":    df_points["within_+-2"].tolist(),
    "full_error": df_points["full_error"].tolist(),
    "exact_match_acc": df_points["exact_match_acc"].tolist()
}"""

In [ ]:
plot_concentration(safe_name, out_dir, results['true'], results['pred'], results['errors'])

In [ ]:
plot_tsne(safe_name, out_dir,
          78,
          sentences=sentences,
          encoder=encoder,
          pca_dims=PCA_DIMS,
          perp=5
          )

In [ ]:
heatmap(results, safe_name, out_dir)

In [ ]:
plot_concentration_color_gradient(results, safe_name, out_dir)

In [ ]:
plot_prediction_histogram(results["pred"], bins=range(0,101,5), model_name=safe_name, out_folder=out_dir)

In [ ]:
plot_error_histogram(results["errors"], safe_name, out_dir, absolute=True)
plot_error_histogram(results["full_error"], safe_name, out_dir, absolute=False)

Histogram is a great way to visualize error, but adding another one on top that is error instead of absolute error, to see if it tends to collapse upwards or downwards, maybe?